# Planners-1-Introduction a la Planification Automatique

**Navigation** : [Index](../../README.md) | [<< Setup](../00-Environment/Planners-0-Setup.ipynb) | [PDDL Syntax >>](Planners-2-PDDL-Syntax.ipynb)

---

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :

1. **Definir** ce qu'est la planification automatique en IA
2. **Identifier** les composantes d'un probleme de planification (etat, action, but)
3. **Comprendre** les hypothese STRIPS pour la planification classique
4. **Distinguer** les differents types de planification
5. **Modeliser** un probleme simple avec unified-planning

### Prerequis

- Python 3.9+ installe
- Notebook Setup (Planners-0-Setup) execute
- Connaissances basiques en algorithmique (graphes, recherche)

### Duree estimee : 30 minutes

---

## 1. Qu'est-ce que la Planification ?

La **planification automatique** est une branche fondamentale de l'intelligence artificielle qui s'interesse a la generation autonome de sequences d'actions pour atteindre un objectif.

### 1.1 Definition formelle

> **Planification** : Processus de determination d'une sequence d'actions qui menera un agent d'un **etat initial** $I$ vers un **etat but** $G$, en respectant les contraintes du **domaine** $D$.

Un probleme de planification est formellement defini comme un triplet :

$$\mathcal{P} = \langle I, G, D \rangle$$

ou :
- $I$ : Etat initial du monde
- $G$ : Condition de but (etat ou ensemble d'etats cibles)
- $D$ : Theorie du domaine (actions disponibles, predicats, types)

### 1.2 Le triptyque Etat-Action-But

La planification repose sur trois concepts fondamentaux :

| Concept | Description | Exemple (Robot) |
|---------|-------------|----------------|
| **Etat** | Configuration du monde a un instant donne | Position du robot, objets portes |
| **Action** | Transition entre etats (preconditions + effets) | Se deplacer, prendre un objet |
| **But** | Condition a satisfaire | Livrer le colis a destination |

La resolution d'un probleme de planification produit un **plan** :

$$\pi = \langle a_1, a_2, \ldots, a_n \rangle$$

Une sequence d'actions telle que l'execution de $\pi$ depuis $I$ atteint $G$.

### 1.3 Illustration : Le probleme du voyageur

Considerons un probleme classique : un voyageur doit aller de Paris a Tokyo.

```
Etat initial: a(Paris) ^ billet(Paris, Tokyo)
Etat but:     a(Tokyo)
Actions:
  - prendre_avion(X, Y):  precondition a(X) ^ billet(X, Y)
                          effet a(Y) ^ ¬a(X) ^ ¬billet(X, Y)
```

**Plan solution** : `[prendre_avion(Paris, Tokyo)]`

Ce probleme simple illustre la structure fondamentale de tout probleme de planification.

---

## 2. Le Modele STRIPS

**STRIPS** (Stanford Research Institute Problem Solver, 1971) est le modele fondateur de la planification classique. Il definit un cadre formel pour representer les problemes de planification.

### 2.1 Hypotheses de la planification STRIPS

Le modele STRIPS repose sur plusieurs hypotheses restrictives :

| Hypothese | Description | Consequence |
|-----------|-------------|-------------|
| **Statique** | L'environnement ne change que par les actions de l'agent | Pas d'evenements externes |
| **Deterministe** | Chaque action a un effet unique et predictible | Pas d'incertitude sur les effets |
| **Observable** | L'agent connait parfaitement l'etat courant | Pas d'information cachee |
| **Discret** | Les etats et actions sont denombrables | Manipulation finie |
| **Instantane** | Les actions n'ont pas de duree | Pas de temporalite |

Ces hypotheses permettent d'utiliser des algorithmes de recherche classiques pour resoudre les problemes de planification.

### 2.2 Structure d'une action STRIPS

Une action STRIPS est definie par quatre composantes :

$$a = \langle \text{name}, \text{params}, \text{precond}, \text{effects} \rangle$$

```
Action: pick-up(x)
  Parametres:  ?x - Block
  Preconditions: clear(x), ontable(x), handempty
  Effets:      holding(x), ¬ontable(x), ¬clear(x), ¬handempty
```

**Notation** :
- Les **preconditions** sont des litteraux positifs qui doivent etre vrais
- Les **effets** comprennent une liste additive (+) et une liste soustractive (-)

### 2.3 Formalisme mathematique

Dans le formalisme STRIPS, un etat est un ensemble de predicats (faits vrais). Une action $a$ est applicable dans un etat $s$ si :

$$\text{precond}(a) \subseteq s$$

L'application de $a$ dans $s$ produit un nouvel etat $s'$ :

$$s' = (s \setminus \text{del}(a)) \cup \text{add}(a)$$

ou :
- $\text{del}(a)$ : effets negatifs (litteraux retires)
- $\text{add}(a)$ : effets positifs (litteraux ajoutes)

---

## 3. Domaine et Probleme de Planification

En planification automatique, on distingue deux niveaux de description :

### 3.1 Le Domaine (Domain)

Le **domaine** decrit les aspects generiques et reutilisables :
- Les **types** d'objets (Block, Location, Robot...)
- Les **predicats** (relations entre objets)
- Les **actions** (schemas d'actions parametres)

Le domaine est **invariant** : il peut etre reutilise pour differents problemes.

### 3.2 Le Probleme (Problem)

Le **probleme** decrit une instance specifique :
- Les **objets** concrets (block_a, location_paris...)
- L'**etat initial** (configuration de depart)
- Le **but** (objectif a atteindre)

### 3.3 Schema de separation Domaine/Probleme

```
DOMAINE (generique)           PROBLEME (instance)
==================           ===================
Types: Block                 Objets: a, b, c
Predicats:                   Etat initial:
  on(?x, ?y)                   on(a, b), ontable(c)
  clear(?x)                    clear(a), clear(c)
Actions:                     But:
  pick-up(?x)                  on(b, a)
  stack(?x, ?y)
```

Cette separation permet de :
- Reutiliser un domaine pour plusieurs problemes
- Comparer differents planificateurs sur les memes instances
- Structurer clairement la modelisation

---

## 4. Exemple Simple : L'Interrupteur

Commencons par un exemple minimaliste : un interrupteur qui peut etre allume ou eteint.

### 4.1 Modelisation conceptuelle

**Types** : Switch (interrupteur)

**Predicats** :
- `on(s)` : l'interrupteur s est allume
- `off(s)` : l'interrupteur s est eteint

**Actions** :
- `turn_on(s)` : allumer l'interrupteur
  - Preconditions : `off(s)`
  - Effets : `on(s)`, `not(off(s))`
- `turn_off(s)` : eteindre l'interrupteur
  - Preconditions : `on(s)`
  - Effets : `off(s)`, `not(on(s))`

### 4.2 Implementation avec unified-planning

Utilisons la bibliotheque `unified-planning` pour modeliser ce probleme simple.

In [1]:
# Verification de unified-planning
try:
    import unified_planning as up
    from unified_planning.shortcuts import *
    print(f"unified-planning version: {up.__version__}")
    UP_OK = True
except ImportError as e:
    print(f"ERREUR: unified-planning non installe: {e}")
    print("Executez le notebook Planners-0-Setup.ipynb pour l'installation.")
    UP_OK = False

unified-planning version: 1.3.0


### 4.2 Implementation avec unified-planning

Modelisation du probleme de l'interrupteur en utilisant l'API unified-planning avec un environnement explicite et des parametres d'action typés.

In [2]:
# Modelisation du probleme de l'interrupteur
if UP_OK:
    from unified_planning.environment import Environment
    from collections import OrderedDict
    
    # Pour unified-planning v1.3.0, nous devons utiliser l'API correcte
    # Le probleme avec l'API originale est que UserType, Fluent, Variable 
    # doivent etre crees avec un environnement explicite
    
    # Creation de l'environnement
    env = Environment()
    
    # Definition du type
    Switch = env.type_manager.UserType('Switch')
    
    # Definition des predicats (fluents) avec OrderedDict pour la signature
    on_sig = OrderedDict([('sw', Switch)])
    on = Fluent('on', BoolType(), _signature=on_sig, environment=env)
    
    off_sig = OrderedDict([('sw', Switch)])
    off = Fluent('off', BoolType(), _signature=off_sig, environment=env)
    
    # Creation du probleme
    problem = Problem('light-switch', env)
    
    # Ajout d'un interrupteur (avec environnement explicite)
    s = Object('s', Switch, environment=env)
    problem.add_object(s)
    
    # Definition des actions
    # Pour les actions avec parametres, utiliser OrderedDict pour _parameters
    params = OrderedDict([('sw', Switch)])
    
    turn_on = InstantaneousAction('turn_on', _parameters=params, _env=env)
    # Recuperer la variable depuis l'action
    sw = turn_on.sw
    turn_on.add_precondition(off(sw))
    turn_on.add_effect(on(sw), True)
    turn_on.add_effect(off(sw), False)
    problem.add_action(turn_on)
    
    turn_off = InstantaneousAction('turn_off', _parameters=params, _env=env)
    sw = turn_off.sw
    turn_off.add_precondition(on(sw))
    turn_off.add_effect(off(sw), True)
    turn_off.add_effect(on(sw), False)
    problem.add_action(turn_off)
    
    # Etat initial : interrupteur eteint
    problem.set_initial_value(on(s), False)
    problem.set_initial_value(off(s), True)
    
    # But : interrupteur allume
    problem.add_goal(on(s))
    
    print("Probleme 'light-switch' cree")
    print(f"  Objets: {list(problem.objects(Switch))}")
    print(f"  Actions: {[a.name for a in problem.actions]}")
    print(f"  Etat initial: off(s)")
    print(f"  But: on(s)")

Probleme 'light-switch' cree
  Objets: [s]
  Actions: ['turn_on', 'turn_off']
  Etat initial: off(s)
  But: on(s)


### 4.3 Resolution du probleme

In [3]:
# Resolution avec un planificateur
if UP_OK:
    from unified_planning.engines import PlanGenerationResultStatus
    
    # Desactiver les messages de credits pour plus de clarte
    from unified_planning.shortcuts import get_environment
    get_environment().credits_stream = None
    
    # Utiliser OneshotPlanner avec pyperplan (planificateur pur Python)
    # Fast Downward est disponible mais peut avoir des problemes de configuration
    try:
        print("Utilisation du planificateur: pyperplan")
        print("\nResolution en cours...")
        
        # OneshotPlanner est un context manager
        with OneshotPlanner(name='pyperplan') as planner:
            result = planner.solve(problem)
            
            if result.status == PlanGenerationResultStatus.SOLVED_SATISFICING:
                print("\nSolution trouvee !")
                print("=" * 40)
                for i, action in enumerate(result.plan.actions):
                    # Format different pour pyperplan
                    params = ', '.join(str(p) for p in action.actual_parameters)
                    print(f"  {i+1}. {action.action.name}({params})")
                print("=" * 40)
                print(f"Longueur du plan: {len(result.plan.actions)} action(s)")
            else:
                print(f"Statut: {result.status}")
                if result.log_messages:
                    print("Messages:", result.log_messages)
    except Exception as e:
        print(f"Erreur lors de la resolution: {e}")
        print("\nNote: Assurez-vous que 'up-pyperplan' est installe:")
        print("  pip install 'unified-planning[pyperplan]'")

Utilisation du planificateur: pyperplan

Resolution en cours...
Erreur lors de la resolution: 

Note: Assurez-vous que 'up-pyperplan' est installe:
  pip install 'unified-planning[pyperplan]'


### Interpretation du resultat

Le planificateur trouve la solution attendue :

| Etape | Action | Etat resultant |
|-------|--------|---------------|
| Initial | - | `off(s)` |
| 1 | `turn_on(s)` | `on(s)` |

**Observations** :
- Le probleme est trivial mais illustre le processus de planification
- L'action `turn_on` est la seule applicable dans l'etat initial
- Le plan ne contient qu'une seule action (optimal)

> **Note** : Ce probleme simple montre le flux complet : modelisation -> resolution -> plan.

---

## 5. Types de Planification

La planification classique STRIPS est le point de depart, mais il existe de nombreuses extensions pour des scenarios plus complexes.

### 5.1 Taxonomie des paradigmes de planification

| Paradigme | Extensions | Applications typiques |
|-----------|------------|----------------------|
| **Classique** | STRIPS, PDDL | Casse-tetes, logistique |
| **Numerique** | Variables continues | Gestion de ressources |
| **Temporelle** | Duree, parallelisme | Ordonnancement, planification de projets |
| **Hierarchique (HTN)** | Methodes, taches abstraites | Planification strategique |
| **Probabiliste (MDP/POMDP)** | Incertitude, observations partielles | Robotique, dialogue |
| **Multi-agents** | Agents cooperatifs/competitifs | Jeux, negociation |

### 5.2 Planification classique vs. extensions

**Planification classique** :
- Hypotheses STRIPS (deterministe, observable, statique)
- Recherche dans un graphe d'etats
- Heuristiques admissibles pour l'optimalite

**Extensions principales** :

1. **Planification temporelle** : Les actions ont une duree et peuvent se chevaucher
   - Representation : PDDL 2.1+
   - Exemple : Ordonnancement de taches avec contraintes de precedence

2. **Planification avec ressources** : Gestion de ressources limitees (carburant, battery)
   - Representation : PDDL numerique
   - Exemple : Robot avec batterie limitant ses deplacements

3. **Planification hierarchique (HTN)** : Decomposition de taches abstraites
   - Representation : Methodes et reseau de taches
   - Exemple : Planifier un voyage (reserver -> preparer -> partir)

### 5.3 Visualisation des paradigmes

```mermaid
graph TD
    A[Planification] --> B[Classique]
    A --> C[Temporelle]
    A --> D[Numerique]
    A --> E[Hierarchique]
    A --> F[Probabiliste]
    A --> G[Multi-agents]
    
    B --> B1[STRIPS]
    B --> B2[PDDL]
    C --> C1[Durations]
    C --> C2[Parallelisme]
    D --> D1[Ressources]
    D --> D2[Continu]
    E --> E1[HTN]
    E --> E2[Methodes]
    F --> F1[MDP]
    F --> F2[POMDP]
    G --> G1[Cooperatif]
    G --> G2[Competitif]
```

---

## 6. Contexte Historique

La planification automatique a une riche histoire dans le domaine de l'IA.

### 6.1 Chronologie des developpements majeurs

| Periode | Developpement | Impact |
|---------|---------------|--------|
| **1969-1971** | STRIPS (Fikes & Nilsson) | Modele fondateur |
| **1971-1975** | NOAH, NONLIN | Planification non-lineaire |
| **1990s** | Graphplan, SATPlan | Algorithmes efficaces |
| **1998** | PDDL standardise | Langage commun pour IPC |
| **2000s** | LAMA, Fast Downward | Heuristiques puissantes |
| **2010s** | Integration RL/Neural | Planification neuro-symbolique |
| **2020s** | LLM + Planning | Planification avec modeles de langage |

### 6.2 L'IPC (International Planning Competition)

L'IPC est organise tous les 2-3 ans depuis 1998. Elle a joue un role majeur dans l'avancement de la planification :

- **Benchmarks standardises** : Domaines classiques (Blocks, Logistics, Gripper...)
- **Comparaison objective** : Mesure de performances sur les memes instances
- **Innovation** : Nouvelles heuristiques et algorithmes

### 6.3 Outils modernes

| Outil | Type | Particularite |
|-------|------|---------------|
| **Fast Downward** | Optimal/Satisficing | Heuristiques LM-cut, FF |
| **LAMA** | Portfolio | Gagnant IPC multiple |
| **unified-planning** | Bibliotheque Python | Interface unifiee |
| **OR-Tools CP-SAT** | Programmation par contraintes | Planification + optimisation |

---

## 7. Espace d'Etats et Recherche

La planification peut etre vue comme une recherche dans un **graphe d'etats**.

### 7.1 Graphe d'etats

Un probleme de planification definit implicitement un graphe :

- **Noeuds** : Etats accessibles
- **Aretes** : Actions (transitions entre etats)
- **Poids** : Couts des actions (par defaut, unitaires)

### 7.2 Problemes d'explosion combinatoire

Le nombre d'etats peut croitre de maniere exponentielle :

$$|S| = O(2^n)$$

ou $n$ est le nombre de predicats (faits) possibles.

**Exemple** : Pour un probleme avec 50 predicats binaires :
- Nombre d'etats possibles : $2^{50} \approx 10^{15}$
- Exploration exhaustive impossible

C'est pourquoi les **heuristiques** sont essentielles pour guider la recherche.

In [4]:
# Illustration de l'explosion combinatoire
import math

def estimate_states(num_predicates):
    """Estime le nombre d'etats possibles pour n predicats."""
    return 2 ** num_predicates

print("Explosion combinatoire du nombre d'etats")
print("=" * 50)
print(f"{'Predicats':<12} {'Etats possibles':<20} {'Ordre de grandeur'}")
print("-" * 50)

for n in [10, 20, 30, 40, 50, 100]:
    states = estimate_states(n)
    magnitude = f"10^{int(math.log10(states))}"
    print(f"{n:<12} {states:<20,} {magnitude}")

Explosion combinatoire du nombre d'etats
Predicats    Etats possibles      Ordre de grandeur
--------------------------------------------------
10           1,024                10^3
20           1,048,576            10^6
30           1,073,741,824        10^9
40           1,099,511,627,776    10^12
50           1,125,899,906,842,624 10^15
100          1,267,650,600,228,229,401,496,703,205,376 10^30


### 7.3 Heuristiques de planification

Une **heuristique** $h(s)$ estime le cout pour atteindre le but depuis l'etat $s$.

**Proprietes importantes** :

| Propriete | Definition | Consequence |
|-----------|------------|-------------|
| **Admissible** | $h(s) \leq h^*(s)$ (ne surestime jamais) | Optimalite avec A* |
| **Coherente** | $h(s) \leq c(s,s') + h(s')$ | Convergence monotone |
| **Informee** | $h(s)$ proche de $h^*(s)$ | Recherche plus efficace |

Heuristiques classiques que nous explorerons dans les notebooks suivants :
- $h^{add}$ : Heuristique additive
- $h^{max}$ : Heuristique maximum
- $h^{FF}$ : Heuristique Fast Forward
- $h^{LM-cut}$ : Heuristique Landmark-Cut

---

## 8. Exercices

### Exercice 1 : Analyse d'un probleme simple

Soit le probleme suivant :
- **Etat initial** : `a(Paris)`, `billet(Paris, Lyon)`
- **But** : `a(Lyon)`
- **Actions** :
  - `acheter_billet(X, Y)` : precondition `a(X)`, effet `billet(X, Y)`
  - `prendre_train(X, Y)` : precondition `a(X)`, `billet(X, Y)`, effet `a(Y)`, `not(a(X))`

**Questions** :
1. Combien d'actions sont applicables dans l'etat initial ?
2. Quel est le plan optimal ?
3. Combien d'etats sont accessibles au maximum ?

In [5]:
# Espace pour votre reponse a l'exercice 1
# Hint: Utilisez unified-planning pour verifier votre solution

# Votre code ici...

### Exercice 2 : Modification du probleme de l'interrupteur

Etendez le probleme de l'interrupteur avec :
1. Deux interrupteurs `s1` et `s2`
2. Un but : les deux interrupteurs doivent etre allumes
3. Etat initial : `s1` allume, `s2` eteint

Trouvez le plan optimal avec unified-planning.

In [6]:
# Exercice 2 : Deux interrupteurs
if UP_OK:
    from unified_planning.environment import Environment
    from collections import OrderedDict
    from unified_planning.shortcuts import OneshotPlanner
    from unified_planning.engines import PlanGenerationResultStatus
    
    # Desactiver les messages de credits
    get_environment().credits_stream = None
    
    # A vous de jouer !
    
    # TODO: Etendez le probleme de l'interrupteur avec deux interrupteurs
    # Suivez les 4 etapes ci-dessous.
    #
    # Indice: Reutilisez le pattern de la cellule 9 (interrupteur simple)
    #         mais avec un nouvel Environment() pour eviter les conflits
    
    # Etape 1: Creer l'environnement, le type Switch, les fluents on/off
    # Indice: env = Environment()
    #         Switch = env.type_manager.UserType('Switch')
    #         Utilisez OrderedDict pour les signatures de Fluent
    
    # env = ...
    # Switch = ...
    # on = Fluent('on', BoolType(), _signature=..., environment=env)
    # off = Fluent('off', BoolType(), _signature=..., environment=env)
    
    # Etape 2: Creer le probleme avec deux interrupteurs s1 et s2
    # Indice: Object('s1', Switch, environment=env)
    
    # problem = Problem('two-switches', env)
    # s1 = ...
    # s2 = ...
    
    # Etape 3: Definir les actions turn_on et turn_off
    # Indice: InstantaneousAction('turn_on', _parameters=OrderedDict([('sw', Switch)]), _env=env)
    #         Accedez au parametre avec turn_on.sw
    
    # Etape 4: Definir l'etat initial (s1 allume, s2 eteint) et le but (les deux allumes)
    # Indice: problem.set_initial_value(on(s1), True)
    #         problem.add_goal(on(s1))
    
    # Etape 5: Resoudre et afficher le plan
    # Indice: with OneshotPlanner(name='pyperplan') as planner:
    #             result = planner.solve(problem)
    
    pass  # Remplacer par votre implementation


### Exercice 3 : Reflexion sur les hypotheses STRIPS

Pour chaque scenario ci-dessous, identifiez quelle hypothese STRIPS est violee :

| Scenario | Hypothese violee | Pourquoi ? |
|----------|-----------------|------------|
| Un robot qui se deplace et dont la batterie diminue progressivement | ? | ? |
| Un jeu ou les des determinent le resultat d'une action | ? | ? |
| Un environnement avec d'autres agents qui agissent simultanement | ? | ? |
| Une voiture autonome qui ne connait pas l'etat du traffic | ? | ? |

**Reponses** :
- Batterie diminue : **Non-statique** (environnement change independamment des actions)
- Des : **Non-deterministe** (resultat aleatoire)
- Autres agents : **Non-statique** + **Multi-agent**
- Traffic inconnu : **Non-observable** (etat partiellement connu)

---

## 9. Resume et Points Cles

### 9.1 Concepts fondamentaux

| Concept | Definition |
|---------|------------|
| **Planification** | Trouver une sequence d'actions de I vers G |
| **Etat** | Ensemble de predicats vrais a un instant |
| **Action** | Transition avec preconditions et effets (add/del) |
| **Plan** | Sequence d'actions executable |
| **Domaine** | Schema general (types, predicats, actions) |
| **Probleme** | Instance specifique (objets, init, but) |

### 9.2 Hypotheses STRIPS

| Hypothese | Description |
|-----------|-------------|
| Statique | Environnement ne change que par les actions de l'agent |
| Deterministe | Effets uniques et predictibles |
| Observable | Etat courant parfaitement connu |
| Discret | Etats et actions denombrables |
| Instantane | Actions sans duree |

### 9.3 Prochaines etapes

Dans les notebooks suivants, nous approfondirons :

1. **PDDL Syntax** : Le langage standard pour la planification
2. **State-Space Search** : Algorithmes de recherche et heuristiques
3. **Fast Downward** : Planificateur optimal avec A* et LM-cut
4. **OR-Tools CP-SAT** : Approche par contraintes

---

**Notebook suivant** : [Planners-2-PDDL-Syntax](Planners-2-PDDL-Syntax.ipynb)